# 06 · Assemble results

Pull together the metrics table (zero-filled vs MAP vs posterior-mean) and
the presentation figures. `viz.panel` saves slide-ready panels.

In [ ]:
import jax.numpy as jnp, numpy as np, matplotlib.pyplot as plt
from mrigen.train_vae import load_model
from mrigen.data import FastMRISlices
from mrigen.fourier import fft2c
from mrigen.masks import equispaced_mask
from mrigen.recon.classical import zero_filled
from mrigen.recon.vae_numpyro import reconstruct_map, reconstruct_posterior
from mrigen import viz, metrics

In [ ]:
vae = load_model('../checkpoints/vae_128.eqx', latent_dim=128)
ds = FastMRISlices('../data/processed')
x = jnp.asarray(ds[0])
mask = equispaced_mask((128, 128), acceleration=4)
y = mask * fft2c(x)

In [ ]:
rows = {}
x_zf = zero_filled(y, mask)
x_map, _ = reconstruct_map(y, mask, vae.decoder, vae.latent_dim)
post = reconstruct_posterior(y, mask, vae.decoder, vae.latent_dim)
for name, rec in [('zero-filled', x_zf), ('MAP', x_map), ('posterior-mean', post['mean'])]:
    rows[name] = dict(PSNR=metrics.psnr(x, rec, data_range=1.0),
                      SSIM=metrics.ssim(x, rec),
                      NMSE=metrics.nmse(x, rec))
import pprint; pprint.pp(rows)

In [ ]:
fig = viz.panel(x, post['mean'], std=post['std'])
fig.savefig('results_panel.png', dpi=150, bbox_inches='tight')